# 01 — The two modalities become tokens

A VLM must turn pixels and words into compatible sequences of vectors. This notebook focuses on tensor shapes and the mechanics, not a particular production processor. Images are often normalized and split into patches; text is tokenized into integer IDs; embeddings map each ID to a vector. Position information is needed because attention alone does not encode sequence order.

For an image `[B, C, H, W]`, patch size `P` yields `N=(H/P)(W/P)` patches. A flattened patch has `C*P*P` scalar values, then a learned linear projection maps it to hidden size `D`. For text `[B, T]`, an embedding table maps IDs to `[B, T, D]`. Padding and causal masks control which keys a query may read.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

torch.manual_seed(7)
B, C, H, W, P, D = 2, 3, 16, 16, 4, 32
images = torch.randn(B, C, H, W)
patches = images.unfold(2, P, P).unfold(3, P, P)
patches = patches.permute(0, 2, 3, 1, 4, 5).reshape(B, -1, C * P * P)
patch_projection = nn.Linear(C * P * P, D)
visual_tokens = patch_projection(patches)
print('image:', tuple(images.shape), 'patches:', tuple(patches.shape), 'visual tokens:', tuple(visual_tokens.shape))
assert patches.shape == (B, 16, 48) and visual_tokens.shape == (B, 16, D)

## Token IDs, padding, and masks

Real systems use a learned/subword tokenizer and special tokens. This intentionally tiny vocabulary makes the mapping visible. The padding mask marks positions that are not real input. In attention, logits have shape `[B, heads, query_length, key_length]`; disallowed keys are set to negative infinity before softmax.

In [ ]:
vocab = {'<pad>': 0, '<bos>': 1, '<eos>': 2, 'red': 3, 'blue': 4, 'block': 5, 'cup': 6}
sentences = [['<bos>', 'red', 'block', '<eos>'], ['<bos>', 'blue', 'cup', '<eos>']]
ids = torch.tensor([[vocab[t] for t in row] for row in sentences])
embedding = nn.Embedding(len(vocab), D, padding_idx=vocab['<pad>'])
text_tokens = embedding(ids)
padding_mask = ids.ne(vocab['<pad>'])
print('ids:', ids.tolist(), 'text embeddings:', tuple(text_tokens.shape), 'valid:', padding_mask.tolist())

In [ ]:
def scaled_dot_product_attention(q, k, v, allowed=None):
    scores = q @ k.transpose(-2, -1) / (q.shape[-1] ** 0.5)
    if allowed is not None:
        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)
    weights = scores.softmax(dim=-1)
    return weights @ v, weights

# A causal mask: token t may attend only to tokens <= t.
T = 5
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
q = k = v = torch.randn(1, 2, T, 8)  # batch, heads, sequence, head_dim
context, weights = scaled_dot_product_attention(q, k, v, causal[None, None])
print('attention output:', tuple(context.shape), 'weights:', tuple(weights.shape))
print('future attention mass:', float(weights[0, 0].triu(1).sum()))
assert torch.allclose(weights[0, 0].triu(1), torch.zeros(T, T))

## Cross-attention: the bridge

Text queries can attend to image keys/values. The output remains text-shaped: `[B, T, D]`. In common decoder VLMs, visual features are projected to the language model width and inserted as tokens or exposed through cross-attention. Image resolution and tiling change the visual token count and therefore compute/memory. A patch is not automatically an object; the model learns useful visual features from training.

In [ ]:
cross_attention = nn.MultiheadAttention(embed_dim=D, num_heads=4, batch_first=True)
aligned_text, cross_weights = cross_attention(text_tokens, visual_tokens, visual_tokens, need_weights=True)
print('text queries:', tuple(text_tokens.shape), 'image keys/values:', tuple(visual_tokens.shape))
print('conditioned text:', tuple(aligned_text.shape), 'attention map:', tuple(cross_weights.shape))
print('Exercises: change P; compute N. Add padding to a sentence and mask it. Compare self-attention and cross-attention tensor lengths.')